# 01_prepare_data

## 목적
- 문제를 '과거 이력으로 다음 주문의 지연 위험을 예측'하는 구조로 고정한다.
- `feature`와 `label`의 시점을 분리해 데이터 누수를 막는다.
- high-frequency threshold와 delay threshold의 근거를 문서화한다.

## 입력 파일
- `../model_base_hv.csv`: 현재 저장소의 source of truth 역할을 하는 고빈도 고객 집계 특징 데이터
- `../model_base_hv.xlsx`: CSV와 같은 데이터를 확인하기 위한 보조 파일
- `TODO`: 원시 Instacart 주문 이력 파일이 복원되면 sequence builder를 추가한다.

## 출력 파일
- 권장: `results/train_ids.csv`, `results/val_ids.csv`, `results/test_ids.csv`
- 권장: `results/feature_schema.csv`, `results/preprocessing_report.md`

## 핵심 규칙
1. `target` 주문은 label 생성용으로만 사용한다.
2. `feature`는 반드시 target 주문 **이전 이력만** 사용한다.
3. 고빈도 고객은 상위 20%로 정의하고, 현재 기준 threshold는 `24`로 둔다.
4. 라벨은 절대 기준형 이진 분류로 두고 `target_gap > 15 -> delay_risk = 1`을 사용한다.
5. split은 사용자 단위 stratified `train / val / test = 70 / 15 / 15`를 권장한다.
6. 현재 데이터는 사용자당 1행이므로 stratified random split도 user-level split으로 해석할 수 있다.

## NOTE
- 현재 `model_base_hv.csv` 기준 `target_gap` 재집계 결과는 `q80 = 15`, `q90 = 24`, `q95 = 30`이다.
- 따라서 `15일` 기준은 q90이 아니라 q80 수준의 조기탐지 기준으로 설명한다.
- q90 기준인 `24일`은 더 엄격한 고위험 지연 기준 후보로 두고, 중간발표에서 교수님 피드백을 받아 최종 기준을 확정한다.

## 현재 확인된 사실
- 원본 전체 고객 수: 206,209
- 원본 전체 주문 수: 3,421,083
- 고빈도 고객 수: 42,499
- 샘플 수: 42,499
- 양성 수: 7,926
- 양성 비율: 약 18.65%
- split 규모: train 29,749 / validation 6,375 / test 6,375
- 현재 구조: 사용자당 1행
- `target_order_number` 범위: 24 ~ 100

## 발표용 체크리스트
- high-frequency threshold = 24의 근거
- delay threshold = 15의 근거
- feature / label 시점 분리 설명
- split 방식과 leakage 방지 규칙
- calendar date 기준 time split은 아니며, 누수 방지의 핵심은 feature/label 시점 분리라는 점
